# dbt 4: Snapshots and history

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/dbt/dbt-04-snapshots.ipynb)

Run dbt locally against DuckDB using a fresh project for each lesson.

In [ ]:
# Setup: install notebook dependencies in Colab.
!pip install -q dbt-core~=1.9 dbt-duckdb~=1.9 duckdb~=2.0

## Setup and sample data

Create a small dbt project and database in a temporary directory.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import duckdb
from dbt.cli.main import dbtRunner

workspace = TemporaryDirectory()
project = Path(workspace.name)
models = project / "models"
models.mkdir()
warehouse = project / "warehouse.duckdb"
(project / "dbt_project.yml").write_text("name: handbook\nversion: '1.0'\nprofile: handbook\n")
(project / "profiles.yml").write_text(
    f"handbook:\n  outputs:\n    dev:\n      type: duckdb\n      path: '{warehouse}'\n      threads: 1\n"
    "  target: dev\n"
)


def dbt(*args, expect_success=True):
    result = dbtRunner().invoke([*args, "--project-dir", str(project), "--profiles-dir", str(project)])
    if result.success != expect_success:
        raise RuntimeError(f"Unexpected dbt result: {result.exception or result.result}")
    print("dbt completed:", result.success)


def query(sql):
    with duckdb.connect(str(warehouse)) as connection:
        print(connection.execute(sql).fetchall())

## Snapshots and history

A dbt snapshot records observed versions of a mutable source. The timestamp strategy uses an updated-at timestamp to detect changes.

**Where it matters**

Keep history that would disappear when a source updates a row in place.

**Watch out for**

A snapshot observes the source when it runs. Multiple changes between runs can collapse into one observed version.

In [ ]:
snapshots = project / "snapshots"
snapshots.mkdir()
(snapshots / "customers.sql").write_text("""
{% snapshot customer_history %}
{{ config(target_schema='main', unique_key='id', strategy='timestamp', updated_at='updated_at') }}
select * from source_customers
{% endsnapshot %}
""")
with duckdb.connect(str(warehouse)) as connection:
    connection.execute("create table source_customers(id integer, tier varchar, updated_at timestamp)")
    connection.execute("insert into source_customers values (1, 'basic', '2026-01-01')")
dbt("snapshot")
with duckdb.connect(str(warehouse)) as connection:
    connection.execute("update source_customers set tier = 'plus', updated_at = '2026-02-01'")
dbt("snapshot")
query("select tier, dbt_valid_to is null as current from customer_history order by dbt_valid_from")

**What to notice:** The basic version is closed and the plus version is current.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

Which predicate finds the current snapshot row in this example?

In [ ]:
# Try your solution here.

In [ ]:
print("dbt_valid_to is null")

## Key takeaways

- Keep history that would disappear when a source updates a row in place.

**Next:** [dbt 5: Replay event-time microbatches](dbt-05-microbatch.ipynb)